In [1]:
import cv2
import pytesseract
import streamlit as st
import re

In [2]:
invoice_image = cv2.imread("sample_images/invoice_Aaron Bergman_36258.jpg") 
cv2.imshow('Invoice',invoice_image)
cv2.waitKey(10000)
cv2.destroyAllWindows()


In [3]:
pytesseract.pytesseract.tesseract_cmd=r"C:\Program Files\Tesseract-OCR\tesseract.exe"

In [4]:
text = pytesseract.image_to_string(invoice_image)
print(text)

SuperStore | N VO | C E

Date:
Bill To: Ship To:
Ship Mode:
Aaron Bergman 98103, Seattle,
Washington, United Balance Due:
States °
Item Quantity
Global Push Button Manager's Chair, Indigo 1 $48.71

Chairs, Furniture, FUR-CH-4421

Subtotal:
Discount (20%):
Shipping:

Total

Notes:

Thanks for your business!

Terms
Order ID : CA-2012-AB10015140-40974

# 36258

Mar 06 2012

First Class

$50.10

48.71
$9.74
11.13

50.10



In [5]:
data = pytesseract.image_to_data(invoice_image,output_type=pytesseract.Output.DICT)
data

{'level': [1,
  2,
  3,
  4,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  2,
  3,
  4,
  5,
  4,
  5,
  5,
  5,
  5,
  4,
  5,
  5,
  4,
  5,
  5,
  5,
  5,
  4,
  5,
  5,
  5,
  5,
  4,
  5,
  5,
  4,
  5,
  5,
  4,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  2,
  3,
  4,
  5,
  5,
  5,
  2,
  3,
  4,
  5,
  4,
  5,
  5,
  4,
  5,
  2,
  3,
  4,
  5,
  2,
  3,
  4,
  5,
  2,
  3,
  4,
  5,
  5,
  5,
  5,
  2,
  3,
  4,
  5,
  4,
  5,
  5,
  5,
  5,
  2,
  3,
  4,
  5,
  2,
  3,
  4,
  5,
  2,
  3,
  4,
  5,
  5,
  2,
  3,
  4,
  5,
  5,
  5,
  2,
  3,
  4,
  5,
  5,
  2,
  3,
  4,
  5,
  2,
  3,
  4,
  5,
  2,
  3,
  4,
  5,
  4,
  5,
  4,
  5,
  2,
  3,
  4,
  5],
 'page_num': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1

In [6]:
n = len(data['text'])
n

131

In [7]:
def extract_and_show_customer_name(image_path):
    """
    Extract customer name from an invoice image.

    The function:
    1. Finds 'Bill To:'
    2. Finds the customer name below it
    3. Draws a bounding box around the name
    4. Displays the extracted customer name on the image
    5. Returns the customer name and processed image
    """

    # Read image
    image = cv2.imread(image_path)

    if image is None:
        raise FileNotFoundError(
            f"Could not read image: {image_path}"
        )

    # OCR
    data = pytesseract.image_to_data(
        image,
        output_type=pytesseract.Output.DICT
    )

    n = len(data["text"])

    # -----------------------------------------
    # Find "Bill To:"
    # -----------------------------------------
    bill_to_index = None

    for i in range(n - 1):

        current = data["text"][i].strip().lower()
        next_text = data["text"][i + 1].strip().lower()

        if (
            current in ["bill", "bill:"]
            and next_text in ["to", "to:"]
        ):
            bill_to_index = i
            break

    if bill_to_index is None:
        print("Bill To not found")
        return None, image

    # Position of "Bill"
    bill_x = data["left"][bill_to_index]
    bill_y = data["top"][bill_to_index]
    bill_h = data["height"][bill_to_index]

    # -----------------------------------------
    # Find text below "Bill To:"
    # -----------------------------------------
    candidates = []

    for i in range(n):

        text = data["text"][i].strip()

        if not text:
            continue

        # OCR confidence
        try:
            confidence = float(data["conf"][i])
        except:
            confidence = 0

        if confidence < 50:
            continue

        x = data["left"][i]
        y = data["top"][i]

        # Text should be below Bill To
        if y > bill_y + bill_h:

            # Not too far vertically
            if y - bill_y < 100:

                # Approximately same horizontal region
                if abs(x - bill_x) < 250:
                    candidates.append(i)

    if not candidates:
        print("Customer name not found")
        return None, image

    # -----------------------------------------
    # Get first line below "Bill To:"
    # -----------------------------------------
    customer_y = min(
        data["top"][i]
        for i in candidates
    )

    customer_words = []

    for i in candidates:

        if abs(data["top"][i] - customer_y) < 10:

            customer_words.append(
                (
                    data["left"][i],
                    data["top"][i],
                    data["width"][i],
                    data["height"][i],
                    data["text"][i].strip()
                )
            )

    # Sort words from left to right
    customer_words.sort(key=lambda x: x[0])

    # -----------------------------------------
    # Extract customer name
    # -----------------------------------------
    customer_name = " ".join(
        word[4]
        for word in customer_words
    )

    # -----------------------------------------
    # Bounding box around customer name
    # -----------------------------------------
    x1 = min(
        word[0]
        for word in customer_words
    )

    y1 = min(
        word[1]
        for word in customer_words
    )

    x2 = max(
        word[0] + word[2]
        for word in customer_words
    )

    y2 = max(
        word[1] + word[3]
        for word in customer_words
    )

    # -----------------------------------------
    # Draw rectangle
    # -----------------------------------------
    cv2.rectangle(
        image,
        (x1 - 5, y1 - 5),
        (x2 + 5, y2 + 5),
        (0, 255, 0),
        3
    )

    # -----------------------------------------
    # Display extracted customer name
    # -----------------------------------------
    cv2.putText(
        image,
        f"Customer: {customer_name}",
        (x1, max(30, y1 - 15)),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 0, 255),
        2
    )

    # ------------------------------------------------
    # Display image 
    # ------------------------------------------------
    cv2.imshow('Invoice',image)
    cv2.waitKey(5000)
    cv2.destroyAllWindows()

    # Return both
    return customer_name, image

In [8]:
customer_name, processed_image = extract_and_show_customer_name(
    "sample_images/invoice_Aaron Bergman_36258.jpg"
)

print("Customer Name:", customer_name)

Customer Name: Aaron Bergman
